In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import Row
from delta import *
from pyspark.sql import functions as F
from pyspark.sql.window import Window

warehouse_location = 'hdfs://hdfs-nn:9000/warehouse'

builder = SparkSession \
    .builder \
    .appName("Fase Gold - Premios Cinema") \
    .config("spark.sql.warehouse.dir", warehouse_location) \
    .config("hive.metastore.uris", "thrift://hive-metastore:9083") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.jars.packages", "io.delta:delta-core_2.12:2.4.0") \
    .enableHiveSupport() \

spark = spark = configure_spark_with_delta_pip(builder).getOrCreate()

In [2]:
spark.sql("DROP TABLE IF EXISTS gold_projeto.fact_premios_cinema")

DataFrame[]

In [4]:
# Criação da tabela no Hive/Metastore para ficar visível no SQL
spark.sql("""
    CREATE EXTERNAL TABLE IF NOT EXISTS gold_projeto.fact_premios_cinema (
    id_film_awards BIGINT,
    fk_content INT,
    fk_cerimony_date STRING,
    n_nominacions INT,
    n_wins INT,
    prize_category STRING,
    winner_name STRING,
    cerimony STRING,
    cerimony_year INT
    )
    USING DELTA
    LOCATION 'hdfs://hdfs-nn:9000/gold/gold_projeto.db/fact_premios_cinema'
""")

DataFrame[]

In [5]:
# Carregar Tabelas Necessárias
df_oscar = spark.table("projeto.oscar_awards")
df_dim_conteudo = spark.table("gold_projeto.dim_conteudo")
df_dim_tempo = spark.table("gold_projeto.dim_tempo")

In [6]:
# Preparar Dados dos Óscares (Transformações)

# - Criar data ficticia (01-01-ANO) para ligar à Dim_Tempo
df_oscar_prep = df_oscar.withColumn(
    "titulo_match", F.lower(F.trim(F.col("film_name")))
).withColumn(
    "data_referencia", F.to_date(F.concat(F.col("award_year"), F.lit("-01-01")))
).withColumn(
    "Qtd_Nomeacoes", F.lit(1) # Cada linha é uma nomeação
).withColumn(
    "Qtd_Vitorias", F.when(F.col("is_winner") == True, 1).otherwise(0)
)

In [7]:
# JOINS para obter as SKs (Chaves Artificiais)

# Join com Dim_Conteudo (Pelo Título)
df_join_conteudo = df_oscar_prep.join(
    df_dim_conteudo,
    (df_oscar_prep.titulo_match == F.lower(F.trim(df_dim_conteudo.title))) & 
    (F.abs(df_oscar_prep.award_year - df_dim_conteudo.release_year) <= 2), 
    "inner"
)

# Join com Dim_Tempo e Seleção Final
df_premios_cinema = df_join_conteudo.join(
    df_dim_tempo,
    df_join_conteudo.data_referencia == df_dim_tempo.data,
    "inner"
).select(
    # --- Chaves Estrangeiras (FKs) ---
    F.col("id_conteudo").alias("fk_content"),
    F.col("id_data").cast("string").alias("fk_cerimony_date"), # Convertido para String conforme schema
    
    # --- Métricas ---
    F.col("Qtd_Nomeacoes").alias("n_nominacions"),
    F.col("Qtd_Vitorias").alias("n_wins"),
    
    # --- Dimensões Degeneradas e Atributos ---
    F.col("award_category").alias("prize_category"),
    F.col("nominee_name").alias("winner_name"),
    F.lit("Academy Awards").alias("cerimony"), # Valor fixo pois a fonte é os Óscares
    F.col("award_year").alias("cerimony_year")
)

In [8]:
#Gerar chave primaria
windowSpec = Window.orderBy("fk_cerimony_date", "fk_content")

df_final = df_premios_cinema.withColumn(
    "id_film_awards", 
    F.row_number().over(windowSpec)
)

In [9]:
df_final.printSchema()
df_final.show()

root
 |-- fk_content: integer (nullable = true)
 |-- fk_cerimony_date: string (nullable = true)
 |-- n_nominacions: integer (nullable = false)
 |-- n_wins: integer (nullable = false)
 |-- prize_category: string (nullable = true)
 |-- winner_name: string (nullable = true)
 |-- cerimony: string (nullable = false)
 |-- cerimony_year: integer (nullable = true)
 |-- id_film_awards: integer (nullable = false)

+----------+----------------+-------------+------+--------------------+-------------------+--------------+-------------+--------------+
|fk_content|fk_cerimony_date|n_nominacions|n_wins|      prize_category|        winner_name|      cerimony|cerimony_year|id_film_awards|
+----------+----------------+-------------+------+--------------------+-------------------+--------------+-------------+--------------+
|      9266|            1025|            1|     0|writing (original...|      robert ardrey|Academy Awards|         1966|             1|
|      9289|            1025|            1|     

In [10]:
from pyspark.sql import functions as F

# Carregar tabelas
df_facto = df_final # Usar o df_final que está em memória
df_conteudo = spark.table("gold_projeto.dim_conteudo")

print("--- Visão Geral da Tabela de Factos ---")
df_facto.select(
    F.count("*").alias("Total_Linhas_Facto"),
    F.sum("n_nominacions").alias("Total_Nomeados_Pessoas"),
    F.sum("n_wins").alias("Total_Vitorias_Estatuetas")
).show()

print("\n--- 2. Top 5 filmes: Estatuetas (Pessoas) vs Categorias (Óscares Reais) ---")
df_top_filmes = df_facto.filter(F.col("n_wins") == 1) \
    .groupBy("fk_content") \
    .agg(
        F.sum("n_wins").alias("Estatuetas_Entregues"), 
        F.countDistinct("prize_category").alias("Categorias_Ganhas")
    )

df_top_filmes.join(df_conteudo, df_top_filmes.fk_content == df_conteudo.id_conteudo) \
    .select(
        F.col("title"), 
        F.col("release_year"),
        F.col("Estatuetas_Entregues"), 
        F.col("Categorias_Ganhas")
    ) \
    .orderBy(F.desc("Categorias_Ganhas")) \
    .show(5, truncate=False)

print("\n--- 3. Top 5 pessoas mais premiadas ---")
df_facto.filter(F.col("n_wins") == 1) \
    .groupBy("winner_name") \
    .agg(F.count("*").alias("Total_Premios")) \
    .orderBy(F.desc("Total_Premios")) \
    .show(5, truncate=False)

print("\n--- 4. Verificação de dados nulos ---")
df_facto.select(
    F.count(F.when(F.col("fk_content").isNull(), 1)).alias("Nulos_FK_Content"),
    F.count(F.when(F.col("fk_cerimony_date").isNull(), 1)).alias("Nulos_FK_Date"),
    F.count(F.when(F.col("id_film_awards").isNull(), 1)).alias("Nulos_PK")
).show()

--- Visão Geral da Tabela de Factos ---
+------------------+----------------------+-------------------------+
|Total_Linhas_Facto|Total_Nomeados_Pessoas|Total_Vitorias_Estatuetas|
+------------------+----------------------+-------------------------+
|              1680|                  1680|                      344|
+------------------+----------------------+-------------------------+


--- 2. Top 5 filmes: Estatuetas (Pessoas) vs Categorias (Óscares Reais) ---
+---------------------------+------------+--------------------+-----------------+
|title                      |release_year|Estatuetas_Entregues|Categorias_Ganhas|
+---------------------------+------------+--------------------+-----------------+
|titanic                    |1997        |23                  |11               |
|the best years of our lives|1946        |8                   |8                |
|shakespeare in love        |1998        |13                  |7                |
|forrest gump               |1994       

In [11]:
df_final.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("hdfs://hdfs-nn:9000/gold/gold_projeto.db/fact_premios_cinema")

In [12]:
# Verificar Resultados
print("Tabela Fact_Premios_Cinema criada com sucesso!")
spark.table("gold_projeto.fact_premios_cinema").show(10, truncate=False)
print(f"Total de registos: {spark.table('gold_projeto.fact_premios_cinema').count()}")

Tabela Fact_Premios_Cinema criada com sucesso!
+----------+----------------+-------------+------+-----------------------------+------------------+--------------+-------------+--------------+
|fk_content|fk_cerimony_date|n_nominacions|n_wins|prize_category               |winner_name       |cerimony      |cerimony_year|id_film_awards|
+----------+----------------+-------------+------+-----------------------------+------------------+--------------+-------------+--------------+
|9266      |1025            |1            |0     |writing (original screenplay)|robert ardrey     |Academy Awards|1966         |1             |
|9289      |1025            |1            |0     |cinematography (color)       |conrad hall       |Academy Awards|1966         |2             |
|9289      |1025            |1            |0     |directing                    |richard brooks    |Academy Awards|1966         |3             |
|9289      |1025            |1            |0     |writing (adapted screenplay) |richard b

In [13]:
spark.stop()